In [4]:
import pandas as pd
import numpy as np

In [2]:
CICFLOWMETER_TO_CIC_COLUMNS = {
    'src_ip': 'Source IP',
    'dst_ip': 'Destination IP',
    'src_port': 'Source Port',
    'dst_port': 'Destination Port',
    'protocol': 'Protocol',
    'timestamp': 'Timestamp',
    'flow_duration': 'Flow Duration',
    'flow_byts_s': 'Flow Bytes/s',
    'flow_pkts_s': 'Flow Packets/s',
    'fwd_pkts_s': 'Fwd Packets/s',
    'bwd_pkts_s': 'Bwd Packets/s',
    'tot_fwd_pkts': 'Total Fwd Packets',
    'tot_bwd_pkts': 'Total Backward Packets',
    'totlen_fwd_pkts': 'Total Length of Fwd Packets',
    'totlen_bwd_pkts': 'Total Length of Bwd Packets',
    'fwd_pkt_len_max': 'Fwd Packet Length Max',
    'fwd_pkt_len_min': 'Fwd Packet Length Min',
    'fwd_pkt_len_mean': 'Fwd Packet Length Mean',
    'fwd_pkt_len_std': 'Fwd Packet Length Std',
    'bwd_pkt_len_max': 'Bwd Packet Length Max',
    'bwd_pkt_len_min': 'Bwd Packet Length Min',
    'bwd_pkt_len_mean': 'Bwd Packet Length Mean',
    'bwd_pkt_len_std': 'Bwd Packet Length Std',
    'pkt_len_max': 'Max Packet Length',
    'pkt_len_min': 'Min Packet Length',
    'pkt_len_mean': 'Packet Length Mean',
    'pkt_len_std': 'Packet Length Std',
    'pkt_len_var': 'Packet Length Variance',
    'fwd_header_len': 'Fwd Header Length',
    'bwd_header_len': 'Bwd Header Length',
    'fwd_seg_size_min': 'min_seg_size_forward',
    'fwd_act_data_pkts': 'act_data_pkt_fwd',
    'flow_iat_mean': 'Flow IAT Mean',
    'flow_iat_max': 'Flow IAT Max',
    'flow_iat_min': 'Flow IAT Min',
    'flow_iat_std': 'Flow IAT Std',
    'fwd_iat_tot': 'Fwd IAT Total',
    'fwd_iat_max': 'Fwd IAT Max',
    'fwd_iat_min': 'Fwd IAT Min',
    'fwd_iat_mean': 'Fwd IAT Mean',
    'fwd_iat_std': 'Fwd IAT Std',
    'bwd_iat_tot': 'Bwd IAT Total',
    'bwd_iat_max': 'Bwd IAT Max',
    'bwd_iat_min': 'Bwd IAT Min',
    'bwd_iat_mean': 'Bwd IAT Mean',
    'bwd_iat_std': 'Bwd IAT Std',
    'fwd_psh_flags': 'Fwd PSH Flags',
    'bwd_psh_flags': 'Bwd PSH Flags',
    'fwd_urg_flags': 'Fwd URG Flags',
    'bwd_urg_flags': 'Bwd URG Flags',
    'fin_flag_cnt': 'FIN Flag Count',
    'syn_flag_cnt': 'SYN Flag Count',
    'rst_flag_cnt': 'RST Flag Count',
    'psh_flag_cnt': 'PSH Flag Count',
    'ack_flag_cnt': 'ACK Flag Count',
    'urg_flag_cnt': 'URG Flag Count',
    'ece_flag_cnt': 'ECE Flag Count',
    'cwr_flag_count': 'CWE Flag Count',   
    'down_up_ratio': 'Down/Up Ratio',
    'pkt_size_avg': 'Average Packet Size',
    'fwd_seg_size_avg': 'Avg Fwd Segment Size',
    'bwd_seg_size_avg': 'Avg Bwd Segment Size',
    'fwd_byts_b_avg': 'Fwd Avg Bytes/Bulk',
    'fwd_pkts_b_avg': 'Fwd Avg Packets/Bulk',
    'fwd_blk_rate_avg': 'Fwd Avg Bulk Rate',
    'bwd_byts_b_avg': 'Bwd Avg Bytes/Bulk',
    'bwd_pkts_b_avg': 'Bwd Avg Packets/Bulk',
    'bwd_blk_rate_avg': 'Bwd Avg Bulk Rate',
    'subflow_fwd_pkts': 'Subflow Fwd Packets',
    'subflow_bwd_pkts': 'Subflow Bwd Packets',
    'subflow_fwd_byts': 'Subflow Fwd Bytes',
    'subflow_bwd_byts': 'Subflow Bwd Bytes',
    'init_fwd_win_byts': 'Init_Win_bytes_forward',
    'init_bwd_win_byts': 'Init_Win_bytes_backward',
    'active_max': 'Active Max',
    'active_min': 'Active Min',
    'active_mean': 'Active Mean',
    'active_std': 'Active Std',
    'idle_max': 'Idle Max',
    'idle_min': 'Idle Min',
    'idle_mean': 'Idle Mean',
    'idle_std': 'Idle Std',
}

wednesday_own = pd.read_csv(r"C:\glassIDS\data\processed\wednesday_own_flows.csv")
wednesday_own = wednesday_own.rename(columns=CICFLOWMETER_TO_CIC_COLUMNS)

In [5]:
wednesday_cic = pd.read_csv(r"C:\glassIDS\data\raw\MachineLearningCVE\Wednesday-workingHours.pcap_ISCX.csv")
wednesday_cic.columns = wednesday_cic.columns.str.strip()   # same leading-space fix

# columns that exist in both, ignoring the 5 identifier columns CIC's pruned version dropped
identifier_cols = ['Source IP', 'Destination IP', 'Source Port', 'Protocol', 'Timestamp']
shared_cols = []
for c in wednesday_own.columns:
    if c in wednesday_cic.columns and c not in identifier_cols:
        shared_cols.append(c)

print(f"Your flows:  {len(wednesday_own):,}")
print(f"CIC's flows: {len(wednesday_cic):,}")   # expect ~691,406 per the CIC paper
print(f"Shared comparable columns: {len(shared_cols)} / {len(shared_cols) + len(identifier_cols)}")

# rough distribution sanity check on a few key features
own_duration_us = wednesday_own['Flow Duration'] * 1_000_000 #converting seconds into microseconds
print("Flow Duration (both in microseconds)")
print("  yours:", own_duration_us.describe()[['mean', '50%', 'max']].to_dict())
print("  CIC's:", wednesday_cic['Flow Duration'].describe()[['mean', '50%', 'max']].to_dict())

for col in ['Total Fwd Packets', 'Flow Bytes/s']:
    own_finite = wednesday_own[col].replace([np.inf, -np.inf], np.nan).dropna()
    cic_finite = wednesday_cic[col].replace([np.inf, -np.inf], np.nan).dropna()
    print(f"\n{col}")
    print("  yours:", own_finite.describe()[['mean', '50%', 'max']].to_dict())
    print("  CIC's:", cic_finite.describe()[['mean', '50%', 'max']].to_dict())

Your flows:  59,771
CIC's flows: 692,703
Shared comparable columns: 77 / 82
Flow Duration (both in microseconds)
  yours: {'mean': 5661310.321577353, '50%': 50338.0, 'max': 166530947.0}
  CIC's: {'mean': 28001680.746361718, '50%': 61437.0, 'max': 119999998.0}

Total Fwd Packets
  yours: {'mean': 42.54176774690067, '50%': 3.0, 'max': 84646.0}
  CIC's: {'mean': 9.556261485802718, '50%': 2.0, 'max': 203943.0}

Flow Bytes/s
  yours: {'mean': 1652824.2264871309, '50%': 9225.801491504575, 'max': 3094000000.0}
  CIC's: {'mean': 1729533.0806950605, '50%': 515.5100196999999, 'max': 2070000000.0}


In [6]:
worst_flow = wednesday_own.loc[wednesday_own['Flow Bytes/s'].idxmax()]
print(worst_flow[['Flow Duration', 'Total Fwd Packets', 'Total Backward Packets',
                   'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Flow Bytes/s']])

Flow Duration                      0.000001
Total Fwd Packets                         2
Total Backward Packets                    1
Total Length of Fwd Packets            3028
Total Length of Bwd Packets              66
Flow Bytes/s                   3094000000.0
Name: 35685, dtype: object


In [7]:
print((wednesday_own['Flow Duration'] < 0.0001).sum())  # flows under 100 microseconds

1879


In [8]:
wednesday_own_clean = wednesday_own[wednesday_own['Flow Duration'] >= 0.0001]
print(f"Dropped {len(wednesday_own) - len(wednesday_own_clean)} rows ({(len(wednesday_own) - len(wednesday_own_clean))/len(wednesday_own):.1%})")
print("Flow Bytes/s median, cleaned:", wednesday_own_clean['Flow Bytes/s'].median())
print("Flow Bytes/s median, CIC's:  ", wednesday_cic['Flow Bytes/s'].median())

Dropped 1879 rows (3.1%)
Flow Bytes/s median, cleaned: 9250.499998419755
Flow Bytes/s median, CIC's:   519.0026125


In [9]:
print("Your capture covers:")
print(" earliest:", wednesday_own['Timestamp'].min())
print(" latest:  ", wednesday_own['Timestamp'].max())
print()
print("A few raw CIC timestamp samples (checking format before parsing):")
print(wednesday_cic['Timestamp'].head(5).tolist())

Your capture covers:
 earliest: 2017-07-05 17:12:42
 latest:   2017-07-05 18:15:36

A few raw CIC timestamp samples (checking format before parsing):


KeyError: 'Timestamp'

In [10]:
wednesday_cic_benign = wednesday_cic[wednesday_cic['Label'] == 'BENIGN']
print(f"CIC benign-only flows: {len(wednesday_cic_benign):,} of {len(wednesday_cic):,} total")

own_finite = wednesday_own['Flow Bytes/s'].replace([np.inf, -np.inf], np.nan).dropna()
cic_benign_finite = wednesday_cic_benign['Flow Bytes/s'].replace([np.inf, -np.inf], np.nan).dropna()

print("yours (after-hours window):", own_finite.describe()[['mean', '50%', 'max']].to_dict())
print("CIC's benign-only:        ", cic_benign_finite.describe()[['mean', '50%', 'max']].to_dict())

CIC benign-only flows: 440,031 of 692,703 total
yours (after-hours window): {'mean': 1652824.2264871309, '50%': 9225.801491504575, 'max': 3094000000.0}
CIC's benign-only:         {'mean': 2433047.1299290294, '50%': 5390.139595, 'max': 2070000000.0}
